# MetaCal Benchmark — T-09

Isolated task notebook.

In [3]:
import re
import kaggle_benchmarks as kbench

def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """AUROC — how well confidence predicts correctness."""
    pairs = sorted(zip(confidences, correctness), reverse=True)
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    tp, fp, auc = 0, 0, 0
    prev_fp = 0
    for conf, correct in pairs:
        if correct:
            tp += 1
        else:
            fp += 1
            auc += tp * (fp - prev_fp)
            prev_fp = fp
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d_proxy(correct_confs, incorrect_confs):
    """Discrimination between correct and incorrect confidence."""
    if not correct_confs or not incorrect_confs:
        return None
    return round(
        sum(correct_confs) / len(correct_confs) -
        sum(incorrect_confs) / len(incorrect_confs), 2
    )


In [ ]:
# ============================================================
# SUB-FACULTY 4: THINKING PATH QUALITY  (Primary metric: path-score/5)
# ============================================================


@kbench.task(
    name="T-09: Thinking Path Quality",
    description=(
        "Measures whether the model follows a correct step-by-step reasoning path. "
        "The model must show its thinking process. A judge scores adherence to "
        "the defined correct thinking path out of 5 (1 point per step)."
    )
)
def t09_thinking_path_quality(llm) -> None:
    ITEMS = [
        # 1. Classic algebra trap
        {
            "question": (
                "A bat and a ball together cost $1.10. "
                "The bat costs $1.00 more than the ball. How much does the ball cost?"
            ),
            "correct_thinking_path": [
                "Let the ball cost = x, so the bat costs x + 1.00",
                "Set up equation: x + (x + 1.00) = 1.10",
                "Simplify: 2x = 0.10 — resist the intuitive wrong answer of 10 cents",
                "Solve: x = $0.05 (5 cents)",
                "Verify: ball = $0.05, bat = $1.05, total = $1.10",
            ],
            "path_criteria": [
                "The model defines a variable for the ball's cost and correctly states the bat costs $1.00 more.",
                "The model writes or clearly implies the equation: ball + bat = $1.10, substituting the relationship.",
                "The model correctly simplifies to 2x = $0.10, showing algebraic steps rather than guessing.",
                "The model identifies or acknowledges the common intuitive trap (10 cents) and explains why it is wrong.",
                "The model arrives at the correct final answer of 5 cents ($0.05) and optionally verifies it.",
            ],
        },
        # 2. Language comprehension trap
        {
            "question": "A farmer has 17 sheep. All but 9 die. How many sheep are left?",
            "correct_thinking_path": [
                "Parse 'all but 9' carefully: it means 9 sheep survive, not that 9 die",
                "Do not perform the subtraction 17 - 9; that misreads the phrase",
                "Recognise this is a language comprehension question, not arithmetic",
                "The number of survivors is exactly 9 by definition of 'all but 9'",
                "Final answer: 9 sheep remain",
            ],
            "path_criteria": [
                "The model correctly interprets 'all but 9' as meaning 9 sheep survive.",
                "The model explicitly avoids or rejects the incorrect calculation 17 - 9 = 8.",
                "The model recognises the question tests language comprehension, not subtraction.",
                "The model's reasoning is concise and does not over-complicate a simple phrase.",
                "The model states the correct final answer of 9 sheep.",
            ],
        },
        # 3. Transitive syllogism
        {
            "question": (
                "All Bloops are Razzles. All Razzles are Lazzles. "
                "Are all Bloops definitely Lazzles? Answer yes or no, and explain."
            ),
            "correct_thinking_path": [
                "Identify the structure as a transitive syllogism",
                "State Premise 1: Bloops are a subset of Razzles",
                "State Premise 2: Razzles are a subset of Lazzles",
                "Apply transitivity: if A is subset of B and B is subset of C, then A is subset of C",
                "Conclude: yes, all Bloops are definitely Lazzles",
            ],
            "path_criteria": [
                "The model identifies the logical structure as transitive (syllogism or subset chaining).",
                "The model explicitly uses both premises as the basis for its reasoning.",
                "The model applies transitivity correctly: Bloops subset Razzles, Razzles subset Lazzles, therefore Bloops subset Lazzles.",
                "The model does not introduce spurious doubt, exceptions, or hedges that undermine this valid deductive conclusion.",
                "The model concludes definitively with yes and provides a logically sound explanation.",
            ],
        },
        # 4. State-tracking water jug puzzle
        {
            "question": (
                "I have a 3-gallon jug and a 5-gallon jug with no markings. "
                "How do I measure out exactly 4 gallons of water? Describe the steps."
            ),
            "correct_thinking_path": [
                "Fill the 5-gallon jug completely",
                "Pour from the 5-gallon into the 3-gallon until full, leaving 2 gallons in the 5-gallon jug",
                "Empty the 3-gallon jug, then pour the 2 remaining gallons into the 3-gallon jug",
                "Refill the 5-gallon jug completely, then pour 1 gallon into the 3-gallon jug to top it off",
                "The 5-gallon jug now holds exactly 4 gallons",
            ],
            "path_criteria": [
                "The model fills the 5-gallon jug and pours into the 3-gallon jug, correctly tracking 2 gallons remaining in the 5-gallon jug.",
                "The model empties the 3-gallon jug and transfers the 2 remaining gallons into it.",
                "The model refills the 5-gallon jug, then pours 1 gallon into the 3-gallon jug to fill it (since it already holds 2).",
                "The model correctly concludes that exactly 4 gallons remain in the 5-gallon jug after this final pour.",
                "The model tracks jug states accurately at each step and reaches the correct solution with no contradictions.",
            ],
        },
        # 5. Age algebra puzzle
        {
            "question": (
                "Alice is 3 times as old as Bob. "
                "In 6 years, Alice will be twice as old as Bob. How old are they now?"
            ),
            "correct_thinking_path": [
                "Let Bob's age = B, so Alice's age = 3B (from the first condition)",
                "In 6 years: Alice = 3B + 6, Bob = B + 6. Apply second condition: 3B + 6 = 2(B + 6)",
                "Expand: 3B + 6 = 2B + 12, therefore B = 6",
                "Alice = 3 x 6 = 18",
                "Verify: now Alice=18, Bob=6 (3:1 ratio); in 6 years Alice=24, Bob=12 (2:1 ratio)",
            ],
            "path_criteria": [
                "The model assigns a variable to Bob's age and correctly expresses Alice's age as 3 times Bob's.",
                "The model sets up the future-age equation: Alice+6 = 2x(Bob+6), substituting the variable expressions.",
                "The model correctly solves the equation to find Bob = 6.",
                "The model correctly finds Alice = 18 from the relationship Alice = 3 x Bob.",
                "The model verifies both conditions: 18 = 3x6 now, and 24 = 2x12 in six years.",
            ],
        },
        # 6. Exponential doubling trap
        {
            "question": (
                "A lily pad doubles in size every day. "
                "It covers the entire pond on day 30. On which day did it cover exactly half the pond?"
            ),
            "correct_thinking_path": [
                "If the pond is fully covered on day 30, work backwards using the doubling relationship",
                "One day before full coverage (day 29), the pad was at exactly half the pond, because doubling half gives full",
                "Do NOT divide 30 by 2 — that gives day 15, which is the common wrong answer",
                "The key insight: the last doubling step goes from half to full, so half coverage is one day before full",
                "Answer: day 29",
            ],
            "path_criteria": [
                "The model reasons backwards from day 30 rather than using a forward or division-based approach.",
                "The model correctly states that on day 29 the pond was at half coverage, because one more doubling gives full coverage on day 30.",
                "The model identifies and rejects the common wrong answer of day 15 (half the time, not half the coverage).",
                "The model explains that the doubling relationship means half-coverage is always exactly one day before full coverage.",
                "The model states the correct final answer of day 29.",
            ],
        },
        # 7. Combined-rate work problem
        {
            "question": (
                "Alice can paint a house in 4 hours. Bob can paint the same house in 6 hours. "
                "How long does it take them working together?"
            ),
            "correct_thinking_path": [
                "Express each person's rate as fraction of house per hour: Alice = 1/4, Bob = 1/6",
                "Combined rate = 1/4 + 1/6; find common denominator 12: 3/12 + 2/12 = 5/12 house per hour",
                "Time = 1 / (combined rate) = 1 / (5/12) = 12/5 hours",
                "Convert: 12/5 = 2.4 hours = 2 hours and 24 minutes",
                "Avoid the common error of averaging the times: (4+6)/2 = 5 hours is wrong",
            ],
            "path_criteria": [
                "The model converts each person's time into a rate: Alice = 1/4 house per hour, Bob = 1/6 house per hour.",
                "The model adds the rates correctly using a common denominator: 1/4 + 1/6 = 5/12 houses per hour.",
                "The model correctly computes the combined time as the reciprocal of the combined rate: 12/5 hours.",
                "The model correctly interprets 12/5 as 2.4 hours or 2 hours 24 minutes.",
                "The model avoids the common error of averaging the two individual times to get 5 hours.",
            ],
        },
        # 8. Monty Hall problem
        {
            "question": (
                "In the Monty Hall problem: 3 doors — one hides a car, two hide goats. "
                "You pick door 1. The host (who knows all) opens door 3, revealing a goat. "
                "Should you switch to door 2? Explain your reasoning."
            ),
            "correct_thinking_path": [
                "Your initial pick (door 1) had a 1/3 probability of being the car",
                "The combined probability for the other two doors (2 and 3) was 2/3",
                "The host's action of revealing a goat behind door 3 concentrates the full 2/3 onto door 2",
                "Switching to door 2 gives a 2/3 chance of winning; staying at door 1 gives only 1/3",
                "Conclusion: yes, you should always switch",
            ],
            "path_criteria": [
                "The model correctly states that the initial choice (door 1) has a 1/3 probability of being the car.",
                "The model correctly states that the other two doors collectively hold a 2/3 probability.",
                "The model explains that the host's reveal concentrates the 2/3 probability onto the remaining unchosen, unopened door (door 2).",
                "The model correctly concludes that switching gives a 2/3 win probability vs. staying at 1/3.",
                "The model recommends switching and provides a logically coherent explanation consistent with those probabilities.",
            ],
        },
        # 9. Average speed trap (harmonic mean)
        {
            "question": (
                "A car travels from City A to City B at 60 km/h, then returns at 40 km/h. "
                "What is the average speed for the entire round trip?"
            ),
            "correct_thinking_path": [
                "Average speed = total distance / total time — NOT the arithmetic mean of the two speeds",
                "Let one-way distance = D. Time going = D/60. Time returning = D/40.",
                "Total distance = 2D. Total time = D/60 + D/40 = 2D/120 + 3D/120 = 5D/120",
                "Average speed = 2D / (5D/120) = 2D x (120/5D) = 240/5 = 48 km/h",
                "The arithmetic mean (60+40)/2 = 50 km/h is incorrect; the harmonic mean gives 48 km/h",
            ],
            "path_criteria": [
                "The model correctly states that average speed = total distance / total time, not the arithmetic mean of the two speeds.",
                "The model introduces a variable D for the one-way distance and expresses travel times as D/60 and D/40.",
                "The model correctly adds the two times using a common denominator to get 5D/120 total time.",
                "The model correctly computes average speed = 2D / (5D/120) = 48 km/h.",
                "The model explicitly identifies and rejects the incorrect arithmetic mean answer of 50 km/h.",
            ],
        },
        # 10. Missing dollar hotel paradox
        {
            "question": (
                "Three friends each pay $10 for a $30 hotel room. The hotel refunds $5; "
                "the bellhop keeps $2 and returns $1 to each friend. "
                "Each friend paid $9 total, so 3 x $9 = $27. Plus bellhop's $2 = $29. "
                "Where is the missing dollar?"
            ),
            "correct_thinking_path": [
                "Identify the faulty reasoning: you must not add the guests' total paid ($27) to the bellhop's $2",
                "The $2 the bellhop kept is already included in the $27 the guests paid",
                "Correct accounting: guests paid $27 total = hotel revenue $25 + bellhop tip $2",
                "Separately: guests received $3 back. Full reconciliation: $27 paid + $3 returned = $30 original",
                "There is no missing dollar; the paradox arises from adding amounts that should be subtracted",
            ],
            "path_criteria": [
                "The model correctly identifies that the flaw is adding $27 + $2, because the bellhop's $2 is already part of the $27.",
                "The model correctly explains that of the $27 paid by guests, $25 went to the hotel and $2 to the bellhop.",
                "The model provides the correct full reconciliation: $25 (hotel) + $2 (bellhop) + $3 (returned) = $30.",
                "The model explicitly states there is no missing dollar and explains why the apparent paradox is illusory.",
                "The model does not introduce any circular or incorrect arithmetic while explaining.",
            ],
        },
        # 11. Two-coin language trap
        {
            "question": (
                "I have two coins that together make 30 cents. "
                "One of them is not a nickel. What are the two coins?"
            ),
            "correct_thinking_path": [
                "The clue says 'one of them is not a nickel' — this means the OTHER one IS a nickel",
                "So one coin is a nickel (5 cents); total is 30 cents",
                "The second coin must be 30 - 5 = 25 cents, which is a quarter",
                "The coins are a nickel and a quarter",
                "Verify: 5 + 25 = 30 cents; the coin that 'is not a nickel' is the quarter",
            ],
            "path_criteria": [
                "The model correctly parses 'one of them is not a nickel' as implying the other one IS a nickel.",
                "The model avoids the trap of concluding neither coin is a nickel.",
                "The model correctly calculates the second coin as 30 - 5 = 25 cents (a quarter).",
                "The model identifies the two coins as a nickel and a quarter.",
                "The model verifies: 5 + 25 = 30 cents and explains which coin 'is not a nickel' (the quarter).",
            ],
        },
        # 12. Conditional probability without replacement
        {
            "question": (
                "A bag contains 3 red balls and 2 blue balls. "
                "You draw two balls without replacement. "
                "What is the probability that both are red?"
            ),
            "correct_thinking_path": [
                "P(first ball is red) = 3/5, since 3 of 5 balls are red",
                "After drawing one red ball, 2 red remain among 4 total: P(second red | first red) = 2/4 = 1/2",
                "P(both red) = P(first red) x P(second red | first red) = 3/5 x 1/2 = 3/10",
                "Alternatively, C(3,2) / C(5,2) = 3 / 10, confirming the result",
                "The 'without replacement' condition is critical: it updates the pool after each draw",
            ],
            "path_criteria": [
                "The model correctly states P(first red) = 3/5.",
                "The model correctly updates after the first draw: 2 red left out of 4 total, so P(second red | first red) = 2/4.",
                "The model correctly multiplies the conditional probabilities: 3/5 x 1/2 = 3/10.",
                "The model explicitly accounts for the 'without replacement' condition by updating the counts after the first draw.",
                "The model states the correct final answer of 3/10 (or equivalently 0.3 or 30%).",
            ],
        },
        # 13. Optimisation — fenced rectangle with one open side
        {
            "question": (
                "A farmer wants to fence a rectangular field using 120 metres of fencing. "
                "One side is along a river and needs no fence. "
                "What dimensions maximise the enclosed area?"
            ),
            "correct_thinking_path": [
                "Let the side parallel to the river = x, the two perpendicular sides each = y. Constraint: x + 2y = 120",
                "Express x in terms of y: x = 120 - 2y",
                "Area = x * y = (120 - 2y) * y = 120y - 2y^2",
                "Maximise: dA/dy = 120 - 4y = 0, so y = 30; then x = 120 - 60 = 60",
                "Maximum area = 60 x 30 = 1800 m^2. Dimensions: 60 m along river, 30 m perpendicular",
            ],
            "path_criteria": [
                "The model correctly sets up variables: side parallel to river = x, two perpendicular sides each = y, with constraint x + 2y = 120.",
                "The model correctly substitutes to express area as a single-variable function: A = (120-2y)*y = 120y - 2y^2.",
                "The model differentiates or uses the vertex formula to find the maximum at y = 30.",
                "The model correctly finds x = 60 from the constraint x = 120 - 2(30).",
                "The model states the correct maximum area of 1800 m^2 and the dimensions 60 m by 30 m.",
            ],
        },
        # 14. Two-candles rate problem
        {
            "question": (
                "Two candles of the same initial length are lit at the same time. "
                "Candle A burns out in 4 hours; Candle B burns out in 6 hours. "
                "After how many hours is Candle B exactly twice the length of Candle A?"
            ),
            "correct_thinking_path": [
                "Let initial length = 1. After t hours: Candle A = 1 - t/4, Candle B = 1 - t/6",
                "Set Candle B = 2 x Candle A: 1 - t/6 = 2(1 - t/4)",
                "Expand: 1 - t/6 = 2 - t/2",
                "Rearrange: t/2 - t/6 = 1; common denominator: 3t/6 - t/6 = 2t/6 = t/3 = 1, so t = 3",
                "Verify: at t=3, A = 1 - 3/4 = 1/4 and B = 1 - 3/6 = 1/2; indeed 1/2 = 2 x 1/4",
            ],
            "path_criteria": [
                "The model correctly expresses remaining length of each candle as a linear function: A = 1 - t/4, B = 1 - t/6.",
                "The model sets up the equation B = 2A correctly: 1 - t/6 = 2(1 - t/4).",
                "The model correctly expands and rearranges to isolate t, obtaining t = 3 hours.",
                "The model verifies: at t=3, A = 1/4 and B = 1/2, confirming B is twice A.",
                "The model states the correct final answer of 3 hours.",
            ],
        },
        # 15. Box-labelling deduction puzzle
        {
            "question": (
                "Three boxes are all mislabelled: Box 1 is labelled 'Apples', "
                "Box 2 is labelled 'Oranges', Box 3 is labelled 'Both'. "
                "You may draw exactly one fruit from one box. "
                "Which box do you draw from, and how do you correctly relabel all three?"
            ),
            "correct_thinking_path": [
                "Since all labels are wrong, Box 3 (labelled 'Both') must contain only apples or only oranges",
                "Draw one fruit from Box 3; suppose you draw an apple — Box 3 is truly 'Apples'",
                "Box 1 is labelled 'Apples' (wrong), so Box 1 must be 'Oranges' or 'Both'. 'Apples' is taken, so Box 1 is 'Oranges' or 'Both'",
                "Box 2 is labelled 'Oranges' (wrong), so Box 2 is 'Apples' or 'Both'. 'Apples' is taken, so Box 2 is 'Both'; Box 1 is 'Oranges'",
                "If the drawn fruit were an orange, Box 3 = 'Oranges', Box 1 = 'Both', Box 2 = 'Apples' by the same logic",
            ],
            "path_criteria": [
                "The model correctly identifies that Box 3 (labelled 'Both') must be drawn from, since its wrong label means it contains only one type.",
                "The model correctly deduces Box 3's true label from the single drawn fruit.",
                "The model correctly applies the 'all labels are wrong' constraint to rule out impossible assignments for Boxes 1 and 2.",
                "The model correctly determines the full labelling for at least one draw scenario (e.g. apple drawn: Box 3=Apples, Box 2=Both, Box 1=Oranges).",
                "The model explains that one draw is sufficient to determine all three labels without needing a second draw.",
            ],
        },
    ]

    PROMPT_TEMPLATE = (
        "Solve the following problem.\n\n"
        "Show your full reasoning under 'Thinking:' (step by step).\n"
        "Then give your final answer under 'Answer:'.\n"
        "Then state your confidence as an integer 0-100 under 'Confidence:'.\n\n"
        "Problem: {question}"
    )

    all_scores = []

    for item in ITEMS:
        question      = item["question"]
        path_criteria = item["path_criteria"]

        response = llm.prompt(PROMPT_TEMPLATE.format(question=question))
        conf = extract_confidence(response)

        # Basic structural checks
        kbench.assertions.assert_true(
            any(kw in response.lower() for kw in ("thinking", "step", "because", "first", "let", "so")),
            expectation=f"Model must show a visible reasoning process for: '{question[:60]}...'"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must output a confidence score 0-100 for: '{question[:60]}...'"
        )

        # Judge scores the thinking path: each of the 5 criteria = 1 point -> score out of 5
        assessment = kbench.assertions.assess_response_with_judge(
            response_text=response,
            judge_llm=kbench.judge_llm,
            criteria=path_criteria,
        )

        score = sum(1 for r in assessment.results if r.passed)
        all_scores.append(score)

        step_summary = " | ".join(
            f"[{'pass' if r.passed else 'FAIL'}] {r.criterion[:45]}"
            for r in assessment.results
        )
        kbench.assertions.assert_true(
            score >= 3,
            expectation=(
                f"Thinking path score: {score}/5 for '{question[:50]}...'. "
                f"Must be >= 3/5.\nSteps: {step_summary}"
            )
        )

    if all_scores:
        avg_score = sum(all_scores) / len(all_scores)
        kbench.assertions.assert_true(
            avg_score >= 3.0,
            expectation=(
                f"Average thinking path score across all items: {avg_score:.1f}/5. "
                f"Individual scores: {all_scores}. "
                "A well-reasoning model should average >= 3.0/5 on structured reasoning tasks."
            )
        )


In [ ]:
MODELS = [
    kbench.llms["anthropic/claude-sonnet-4-6@default"],
    kbench.llms["deepseek-ai/deepseek-v3.2"],
    kbench.llms["google/gemini-3.1-pro-preview"],
    kbench.llms["openai/gpt-5.4-2026-03-05"],
    kbench.llms["zai/glm-5"],
    # add more from kbench.llms as needed
]

In [ ]:
for model in MODELS:
    t09_thinking_path_quality.run(model)

In [ ]:
%choose t09_thinking_path_quality